# LN, autoencoder, and subunit-model tutorial

This tutorial fits an LN baseline, trains the spike-triggered autoencoder, examines its encoder weights and Moran-selected subunits, builds the derived rectified subunit model, and compares both models on frozen noise. Run it from the repository root with the `rgc_subunit_classifier` Conda kernel.

In [ ]:
from pathlib import Path
import json
import numpy as np
from matplotlib import pyplot as plt
from scipy.io import loadmat
from src.analysis.model_comparison import prediction_metrics
from src.prediction.sta_model_predictions import gen_sta_model, get_sta_predictions
from src.prediction.subunit_model_predictions import gen_subunit_model, get_subunit_predictions
from src.simulation.generate_training_data import crop_sta, gen_sta
from src.simulation.run_autoencoder_pipeline import run_autoencoder_model
from src.visualization.plotting_fxns import plot_predictions, plot_subunits

## Configuration

These are full-analysis defaults. For a smoke demonstration only, use `MAX_TRIALS = 2`, `NODE_NUM = 3`, `EPOCHS = 1`, and `MORANS_THRESHOLD = -1`. Smoke outputs verify software wiring and are not scientific results.

In [ ]:
CELL = 0
DATA_PATH = Path('data/cell_data_01_NC.mat')
STIMULUS_PATH = Path('data/stimulus_data')
STA_PATH = Path('results/sta')
NODE_NUM, EPOCHS, BATCH_SIZE = 60, 100, 100
LEARNING_RATE, L1_COEFFICIENT, L2_COEFFICIENT = 0.001, 1e-4, 0
SPATIAL_COEFFICIENT = 0
OUTPUT_ACTIVATION = 'sigmoid'  # use 'linear' as a controlled alternative
SIGMA, MORANS_THRESHOLD, RANDOM_STATE = 3, 0.25, 0
MAX_TRIALS = None
WIDTH, HEIGHT, REFRESH_RATE = 200, 150, 30
FILTER_LENGTH = int(0.67 * REFRESH_RATE)

## 1. Load the STA and receptive-field crop

The autoencoder and LN model use the same STA-derived temporal filter and spatial crop, keeping the downstream comparison aligned.

In [ ]:
spikecounts = loadmat(DATA_PATH)['spk1'][CELL]
num_frames, total_trials = spikecounts.shape
num_trials = total_trials if MAX_TRIALS is None else min(MAX_TRIALS, total_trials)
spikecounts = spikecounts[:, :num_trials]
names = {
    'spatial_sta_filename': f'Cell {CELL} Uncropped Spatial STA.h5',
    'temp_sta_filename': f'Cell {CELL} Uncropped Temp STA.h5',
    'sta_filename': f'Cell {CELL} Uncropped STA.h5',
}
sta, spatial_sta, temporal_sta = gen_sta(
    WIDTH, HEIGHT, spikecounts, CELL, FILTER_LENGTH, num_frames, num_trials,
    sta_path=STA_PATH, stim_path=STIMULUS_PATH, **names,
)
cropped_sta, crop, ellipse_mask = crop_sta(spatial_sta, SIGMA)
crop_shape = cropped_sta.shape
stac = sta.reshape(WIDTH, HEIGHT, FILTER_LENGTH)[crop].reshape(cropped_sta.size, FILTER_LENGTH)
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
axes[0].imshow(spatial_sta.T, origin='lower', cmap='bwr'); axes[0].set_title('Spatial STA')
axes[1].plot(np.arange(FILTER_LENGTH) / REFRESH_RATE, temporal_sta); axes[1].set_title('Temporal STA')
axes[2].imshow(cropped_sta.T, origin='lower', cmap='bwr'); axes[2].set_title(f'Crop {crop_shape}')
fig.tight_layout()

## 2. Fit the LN baseline

The STA filters running-noise histories and a softplus nonlinearity maps the projection to firing rate. Frozen-noise prediction is held out from this fit.

In [ ]:
ln_parameters = gen_sta_model(
    WIDTH, HEIGHT, spikecounts, crop, *crop_shape, FILTER_LENGTH,
    num_frames, num_trials, stac, STIMULUS_PATH,
)
ln_predictions, ln_correlation, actual_firing_rate, time = get_sta_predictions(
    CELL, ln_parameters, stac, crop, *crop_shape, DATA_PATH, STIMULUS_PATH,
    filter_length=FILTER_LENGTH,
)
ln_metrics = prediction_metrics(actual_firing_rate, ln_predictions, 3 + stac.size, FILTER_LENGTH)
print(ln_metrics)
plot_predictions(ln_predictions, actual_firing_rate, time, FILTER_LENGTH, label='LN')

## 3. Train the autoencoder and examine encoder weights

Unlike the classifier, the autoencoder sees only spike-triggered ensembles and reconstructs them without spike/no-spike labels. Encoder weights are expanded into the same spatial representation as classifier weights, then filtered using the same Moran's-I rule.

In [ ]:
autoencoder = run_autoencoder_model(
    CELL, DATA_PATH, STIMULUS_PATH, sta_path=STA_PATH, node_num=NODE_NUM,
    batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, num_epochs=EPOCHS,
    l1_coefficient=L1_COEFFICIENT, l2_coefficient=L2_COEFFICIENT,
    spatial_coefficient=SPATIAL_COEFFICIENT, output_activation=OUTPUT_ACTIVATION,
    sigma=SIGMA, random_state=RANDOM_STATE, max_trials=MAX_TRIALS,
    morans_threshold=MORANS_THRESHOLD,
)
print(f'Validation loss: {autoencoder.validation_loss:.5f}')
print(f'Selected {len(autoencoder.subunits)} of {NODE_NUM} encoder nodes')
plot_subunits(autoencoder.node_weights, crop_shape, columns=4);

In [ ]:
if len(autoencoder.subunits) == 0:
    raise RuntimeError('No encoder weights exceeded the Moran threshold; inspect/tune before continuing.')
plot_subunits(autoencoder.subunits, crop_shape, columns=3);

## 4. Build the derived subunit model

Selected encoder filters are normalized, rectified, linearly combined to approximate the spatial STA, and passed through a fitted softplus output nonlinearity.

In [ ]:
subunit_parameters, combination_weights, normalized_subunits = gen_subunit_model(
    autoencoder.subunits, WIDTH, HEIGHT, spikecounts, FILTER_LENGTH,
    num_frames, num_trials, spatial_sta, temporal_sta, crop, *crop_shape, STIMULUS_PATH,
)
subunit_predictions, subunit_correlation, _, _ = get_subunit_predictions(
    CELL, subunit_parameters, combination_weights, normalized_subunits, temporal_sta,
    crop, *crop_shape, DATA_PATH, STIMULUS_PATH, filter_length=FILTER_LENGTH,
)
subunit_metrics = prediction_metrics(
    actual_firing_rate, subunit_predictions, 3 + len(combination_weights), FILTER_LENGTH,
)
print(subunit_metrics)

## 5. Compare frozen-noise performance

Correlation and MSE compare predictions on identical time bins. AIC/BIC are also shown, but their interpretation requires an explicit convention for counting learned spatial-filter parameters.

In [ ]:
comparison = {
    'LN': {'correlation': ln_metrics.correlation, 'mse': ln_metrics.mse},
    'Autoencoder subunit': {
        'correlation': subunit_metrics.correlation, 'mse': subunit_metrics.mse,
    },
}
print(json.dumps(comparison, indent=2))
fig, ax = plt.subplots(figsize=(11, 4))
plot_predictions(ln_predictions, actual_firing_rate, time, FILTER_LENGTH, label='LN', ax=ax)
ax.plot(time[FILTER_LENGTH:], subunit_predictions, label='Autoencoder subunit model')
ax.legend(); fig.tight_layout()

## Interpretation checklist

- Compare sigmoid and linear decoder outputs as explicit model variants because filtered ensembles contain negative values.
- Repeat several seeds and quantify layout stability.
- Tune node count and regularization using held-out behavior.
- Inspect all encoder weights as well as the Moran-selected subset.
- Report crop sigma, threshold, trial count, seed, and optimization settings.
- Never interpret a smoke demonstration as a fitted scientific result.